# AI & LLM Open Source Ecosystem Analysis

**Data Sources:**
- GitHub REST API - Top AI/ML repositories by stars
- HuggingFace API - Most downloaded models

**Objective:** Map the AI open-source landscape, identify dominant frameworks, trending research areas, and community engagement patterns.

---

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({'figure.figsize': (12,6), 'axes.titlesize': 14, 'axes.titleweight': 'bold'})

In [ ]:
gh = pd.read_csv('../data/github_ai_repos.csv')
hf = pd.read_csv('../data/huggingface_models.csv')

print(f'GitHub repos: {len(gh):,}')
print(f'HuggingFace models: {len(hf):,}')
print(f'\nGitHub columns: {list(gh.columns)}')
gh.head()

## 2. GitHub AI Repos - Landscape Overview

In [ ]:
# Top 20 repos by stars
top20 = gh.nlargest(20, 'stars')[['full_name','stars','forks','language','created_at']]

fig, ax = plt.subplots(figsize=(12,8))
bars = ax.barh(top20['full_name'][::-1], top20['stars'][::-1], color=sns.color_palette('viridis', 20))
ax.set_xlabel('Stars'); ax.set_title('Top 20 AI/ML Repositories on GitHub')
for i, (name, stars) in enumerate(zip(top20['full_name'][::-1], top20['stars'][::-1])):
    ax.text(stars + max(top20['stars'])*0.01, i, f'{stars:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/top_repos.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Programming Language Distribution

In [ ]:
lang_stats = gh.groupby('language').agg(
    repos=('repo_name','count'),
    total_stars=('stars','sum'),
    avg_stars=('stars','mean'),
    avg_forks=('forks','mean')
).sort_values('repos', ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(16,6))

lang_stats['repos'].sort_values().plot(kind='barh', ax=axes[0], color='#4C72B0', alpha=0.8)
axes[0].set_xlabel('Number of Repos'); axes[0].set_title('AI Repos by Language')

lang_stats['total_stars'].sort_values().plot(kind='barh', ax=axes[1], color='#55A868', alpha=0.8)
axes[1].set_xlabel('Total Stars'); axes[1].set_title('Total Stars by Language')

plt.tight_layout()
plt.savefig('../outputs/language_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(lang_stats.round(0).to_string())

## 4. Repository Growth Timeline

In [ ]:
gh['created_at'] = pd.to_datetime(gh['created_at'])
gh['created_year'] = gh['created_at'].dt.year

yearly = gh.groupby('created_year').agg(
    repos=('repo_name','count'),
    avg_stars=('stars','mean')
).reset_index()
yearly = yearly[yearly['created_year'] >= 2015]

fig, ax1 = plt.subplots(figsize=(12,6))
ax2 = ax1.twinx()
ax1.bar(yearly['created_year'], yearly['repos'], color='#4C72B0', alpha=0.7, label='New Repos')
ax2.plot(yearly['created_year'], yearly['avg_stars'], color='#DD8452', marker='o', lw=2, label='Avg Stars')
ax1.set_xlabel('Year'); ax1.set_ylabel('New Repos Created', color='#4C72B0')
ax2.set_ylabel('Avg Stars', color='#DD8452')
plt.title('AI Repository Growth Over Time')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/growth_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Stars vs Forks - Community Engagement

In [ ]:
fig, ax = plt.subplots(figsize=(12,7))
scatter = ax.scatter(gh['stars'], gh['forks'], alpha=0.5, s=30,
                     c=gh['created_year'], cmap='viridis', edgecolors='none')
plt.colorbar(scatter, label='Year Created')

# Label top outliers
for _, r in gh.nlargest(8, 'stars').iterrows():
    ax.annotate(r['repo_name'], (r['stars'], r['forks']),
               fontsize=8, alpha=0.8, textcoords='offset points', xytext=(5,5))

ax.set_xlabel('Stars'); ax.set_ylabel('Forks')
ax.set_title('Community Engagement: Stars vs Forks')
ax.set_xscale('log'); ax.set_yscale('log')
plt.tight_layout()
plt.savefig('../outputs/stars_vs_forks.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. License Distribution

In [ ]:
lic = gh['license'].value_counts().head(8)
fig, ax = plt.subplots(figsize=(10,6))
lic.sort_values().plot(kind='barh', ax=ax, color=sns.color_palette('Set2', len(lic)))
ax.set_xlabel('Number of Repos'); ax.set_title('Most Common Licenses in AI Repos')
plt.tight_layout()
plt.savefig('../outputs/license_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. HuggingFace Model Ecosystem

In [ ]:
# Top models by downloads
hf['downloads'] = pd.to_numeric(hf['downloads'], errors='coerce').fillna(0)
hf['likes'] = pd.to_numeric(hf['likes'], errors='coerce').fillna(0)

top_models = hf.nlargest(15, 'downloads')[['model_id','pipeline_tag','downloads','likes']]
top_models['downloads_M'] = (top_models['downloads'] / 1e6).round(1)

fig, ax = plt.subplots(figsize=(12,7))
ax.barh(top_models['model_id'][::-1], top_models['downloads_M'][::-1],
        color=sns.color_palette('magma', 15))
ax.set_xlabel('Downloads (Millions)'); ax.set_title('Top 15 HuggingFace Models by Downloads')
plt.tight_layout()
plt.savefig('../outputs/top_hf_models.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Models by task/pipeline
task_stats = hf.groupby('pipeline_tag').agg(
    count=('model_id','count'),
    total_downloads=('downloads','sum'),
    avg_likes=('likes','mean')
).sort_values('count', ascending=False).head(12)

fig, axes = plt.subplots(1, 2, figsize=(16,6))
task_stats['count'].sort_values().plot(kind='barh', ax=axes[0], color='#4C72B0')
axes[0].set_xlabel('Model Count'); axes[0].set_title('Models by Task Type')
task_stats['total_downloads'].sort_values().plot(kind='barh', ax=axes[1], color='#55A868')
axes[1].set_xlabel('Total Downloads'); axes[1].set_title('Downloads by Task Type')
plt.tight_layout()
plt.savefig('../outputs/hf_task_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. HuggingFace - Downloads vs Likes

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(hf['downloads'], hf['likes'], alpha=0.5, s=30, color='#8172B2')
for _, r in hf.nlargest(5, 'downloads').iterrows():
    ax.annotate(r['model_id'].split('/')[-1], (r['downloads'], r['likes']),
               fontsize=8, textcoords='offset points', xytext=(5,5))
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Downloads (log)'); ax.set_ylabel('Likes (log)')
ax.set_title('Model Popularity: Downloads vs Community Likes')
plt.tight_layout()
plt.savefig('../outputs/hf_downloads_likes.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Key Findings

### GitHub Landscape
1. **Python dominates** the AI ecosystem by a massive margin, followed by Jupyter Notebooks and C++
2. **Explosive growth post-2022** - the LLM revolution drove unprecedented repo creation
3. **Community engagement** follows a power law - a few repos get massive stars while most stay small
4. **Apache-2.0 and MIT** are the dominant licenses, reflecting the open-source AI movement

### HuggingFace Trends
1. **Text generation and embeddings** dominate downloads - driven by LLM/RAG applications
2. **Sentence transformers** are the most downloaded category - showing the importance of semantic search
3. **Downloads vs likes diverge** - utility models get downloads, flashy models get likes

### Implications
- Python + HuggingFace is the de facto AI stack
- LLM tooling (LangChain, Ollama, vLLM) is the fastest-growing category
- Open-weight models are driving democratization of AI

---
*Data sourced live from GitHub REST API & HuggingFace API*